# LoRA Adapter Evaluation — Val Accuracy

Greedy-decodes one or more fine-tuned Nemotron-3-Nano-30B-A3B LoRA adapters over a
**validation parquet** (`problem_id`, `prompt`, `answer` columns) and reports exact-match accuracy.

**What this notebook does:**
1. Loads the base model **once** via vLLM (`enable_lora=True`).
2. Builds chat-template prompts from `prompt` column (once — reused across adapters).
3. For each adapter in `ADAPTERS`:
   a. Generates greedily (temperature=0) using `LoRARequest` — no model reload.
   b. Extracts answers from `\boxed{}` with `extract_final_answer` (official metric logic).
   c. Computes accuracy vs. the `answer` column using `verify`.
   d. Saves per-sample results to `val_generations__<name>.parquet`.
4. Prints a cross-adapter accuracy summary.

**Run target: Kaggle GPU** (30B MoE, vLLM, requires A100/H100).  
Required dataset inputs attached to this notebook:
- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- One or more adapter datasets (each containing `adapter_config.json` + `adapter_model.safetensors`)
- Your val parquet (containing `problem_id`, `prompt`, `answer` columns)

In [ ]:
# ── 0. Kaggle environment bootstrap (no-op outside Kaggle) ─────────────────────
import os, subprocess, sys

ON_KAGGLE = os.path.exists('/kaggle')

if ON_KAGGLE:
    commands = [
        'tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell',
    ]
    for cmd in commands:
        print(f'Running: {cmd}')
        subprocess.run(cmd, shell=True, check=True)
    sys.path.insert(0, '/tmp')

    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['TRANSFORMERS_NO_FLAX'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'
    print('Kaggle env configured.')

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────────
import glob
from pathlib import Path

# ---- Model -----------------------------------------------------------------
if ON_KAGGLE:
    import kagglehub
    MODEL_PATH = kagglehub.model_download(
        'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'
    )
else:
    MODEL_PATH = '/path/to/nemotron-3-nano-30b-a3b-bf16'  # override locally

# ---- Adapters ---------------------------------------------------------------
# Dict of {friendly_name: adapter_directory_path}.
# Each entry writes its own output file: val_generations__<name>.parquet
# Leave empty to auto-discover all adapter_config.json files under /kaggle/input.
ADAPTERS = {
    "dataset-0-examples": "/kaggle/input/models/zosov07/model-dataset-0-examples/transformers/default/1",
    # "dataset-1-examples": "/kaggle/input/models/zosov07/model-dataset-1-examples/transformers/default/1",
}

if not ADAPTERS:
    _configs = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
    if not _configs:
        raise FileNotFoundError(
            'adapter_config.json not found under /kaggle/input. '
            'Attach your adapter dataset or populate ADAPTERS.'
        )
    ADAPTERS = {Path(p).parent.name: str(Path(p).parent) for p in _configs}

print(f'Adapters to evaluate ({len(ADAPTERS)}):')
for name, path in ADAPTERS.items():
    print(f'  {name!r:40s}  →  {path}')

# ---- Validation data -------------------------------------------------------
# Required columns: problem_id, prompt, answer.
VAL_PARQUET_OVERRIDE = "/kaggle/input/datasets/zosov07/solver-results-base/solver_results_base.parquet"

if VAL_PARQUET_OVERRIDE:
    VAL_PARQUET = VAL_PARQUET_OVERRIDE
else:
    _parquets = glob.glob('/kaggle/input/**/val.parquet', recursive=True)
    if not _parquets:
        raise FileNotFoundError(
            'val.parquet not found under /kaggle/input. '
            'Attach your val dataset or set VAL_PARQUET_OVERRIDE.'
        )
    VAL_PARQUET = _parquets[0]

print(f'Val parquet: {VAL_PARQUET}')

# ---- Output ----------------------------------------------------------------
# One file per adapter:  <OUT_DIR>/val_generations__<name>.parquet
OUT_DIR = Path('/kaggle/working') if ON_KAGGLE else Path('data')

def out_path_for(name: str) -> Path:
    """Return the output parquet path for an adapter, sanitising the name."""
    import re as _re
    safe = _re.sub(r'[\s/\\]', '_', name)
    return OUT_DIR / f'val_generations__{safe}.parquet'

# ---- Generation knobs -------------------------------------------------------
MAX_LORA_RANK          = 32
MAX_MODEL_LEN          = 8192
MAX_TOKENS             = 7680   # leaves 512 for the prompt at MAX_MODEL_LEN=8192
TEMPERATURE            = 0.0    # greedy decoding — deterministic, reproducible
TOP_P                  = 1.0
GPU_MEMORY_UTILIZATION = 0.85
MAX_NUM_SEQS           = 64

RANDOM_SEED = 42
print('Config OK.')

## Helper functions

`extract_final_answer` and `verify` are copied verbatim from the official competition metric
(`nvidia-nemotron-metric.ipynb` cell 0) so our scores are directly comparable to the leaderboard.

In [ ]:
# ── 2. Helper functions ─────────────────────────────────────────────────────────
import math
import re


# ----- Copied verbatim from nvidia-nemotron-metric.ipynb cell 0 ---------------

def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return 'NOT_FOUND'

    # Search for boxed answer. For each \boxed{ occurrence, take everything up
    # to the last } before the next \boxed{ (or end of text). This handles
    # answers that themselves contain '}' (the model writes them literally,
    # producing e.g. \boxed{}52} for the answer "}52") as well as nested LaTeX
    # like \boxed{\frac{1}{2}}.
    boxed_starts = list(re.finditer(r'\\boxed\{', text))
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind('}')
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        found = re.findall(pattern, text, re.IGNORECASE)
        if found:
            return found[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    found = re.findall(r'-?\d+(?:\.\d+)?', text)
    if found:
        return found[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify if the answer matches.

    For numerical answers, allow them to be judged as equal within a certain relative tolerance (1e-2);
    otherwise, compare strictly as strings (case-insensitive).

    Examples:
        >>> verify("10011000", "10011000")
        True
        >>> verify("10011000", "10011001")
        False
        >>> verify("24.64", "24.6401")
        True
        >>> verify("XLVII", "xlvii")
        True
        >>> verify("11011", "00011011")
        False
    """
    # Clean up strings
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    # If the answer is a binary string, compare strictly as strings
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()

    try:
        # Try to convert the answers to floating point numbers
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        # Use a small absolute tolerance for numbers near zero
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        # Fallback to case-insensitive string comparison
        return predicted.lower() == stored_answer.lower()


# ----- Additional helpers for this evaluation notebook -----------------------

_COT_MARKERS = [
    'Formatted equation:',
    '**Summary:**',
    'Digit mapping:',
    'solution:',
    'STRATEGIES',
]


def emits_cot_template(text: str) -> bool:
    """Return True if the generation contains cryptarithm CoT template markers.

    Signals over-fitting / catastrophic forgetting when True on a non-symbolic prompt.
    """
    if text is None:
        return False
    return sum(1 for m in _COT_MARKERS if m in text) >= 2


def parse_solution_line(text: str | None) -> str | None:
    """Extract the last 'solution: <answer>' from a generation.

    Returns the answer string (stripped) or None if not present.
    This is the trained-corpus answer format (as opposed to \\boxed{}).
    """
    if not text:
        return None
    hits = re.findall(r'solution:\s*(.+)', text)
    return hits[-1].strip() if hits else None


def build_prompt(tokenizer, prompt_text: str) -> str:
    """Wrap a raw puzzle prompt in the chat template the metric uses."""
    user_content = (
        prompt_text
        + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    )
    try:
        return tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        return user_content


print('Helpers defined.')

## Load validation dataset

In [ ]:
# ── 3. Load validation dataset ───────────────────────────────────────────────────
import pandas as pd

val_df = pd.read_parquet(VAL_PARQUET)

# Guard: required columns
_required = {'problem_id', 'prompt', 'answer'}
_missing  = _required - set(val_df.columns)
assert not _missing, f'Missing required columns: {_missing}'

print(f'Val rows    : {len(val_df):,}')
print(f'Val columns : {list(val_df.columns)}')
print(f'\nFirst 3 ids : {list(val_df["problem_id"].head(3))}')
print(f'Answer sample: {list(val_df["answer"].head(3))}')

## Build the vLLM engine (once — reused across all adapters)

In [ ]:
# ── 4. Build vLLM engine ────────────────────────────────────────────────────────
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=MAX_NUM_SEQS,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=MAX_LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)
tokenizer = llm.get_tokenizer()
print('vLLM engine ready.')

---
## Val Accuracy — greedy decoding over the full val set

In [ ]:
# ── 5. Build prompts for the full val set ───────────────────────────────────────
eval_prompts = [
    build_prompt(tokenizer, str(row['prompt']))
    for _, row in val_df.iterrows()
]
print(f'Built {len(eval_prompts):,} prompts.')

In [ ]:
# ── 6–9. Generate, score, and save — one iteration per adapter ─────────────────
eval_sampling = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
)

# Optional columns passed through from val_df if present in the input parquet.
_OPTIONAL_COLS = [
    'solver_strategy', 'solver_category', 'problem_category',
    'ops_mapping', 'digits_mapping', 'solver_tier', 'solver_radix',
    'ops_candidates', 'cot',
]
_present_optional = [c for c in _OPTIONAL_COLS if c in val_df.columns]
if _present_optional:
    print(f'Optional columns found: {_present_optional}')
else:
    print('No optional columns found in val parquet.')

_core_cols = ['problem_id', 'prompt', 'answer', 'solver_answer', 'raw_output', 'has_boxed', 'correct']

summary_rows = []

for adapter_idx, (adapter_name, adapter_path) in enumerate(ADAPTERS.items()):
    print(f'\n{"="*60}')
    print(f'  Adapter {adapter_idx+1}/{len(ADAPTERS)}: {adapter_name!r}')
    print(f'  Path: {adapter_path}')
    print(f'{"="*60}')

    # ── Generate ─────────────────────────────────────────────────────────────
    eval_outputs = llm.generate(
        eval_prompts,
        sampling_params=eval_sampling,
        lora_request=LoRARequest(adapter_name, adapter_idx + 1, adapter_path),
    )
    print(f'Generation done: {len(eval_outputs):,} completions.')

    # ── Score ────────────────────────────────────────────────────────────────
    results = []
    for row, output in zip(val_df.itertuples(index=False), eval_outputs):
        raw           = output.outputs[0].text
        solver_answer = extract_final_answer(raw)
        has_boxed     = bool(re.search(r'\\boxed\{', raw))
        ground_truth  = str(row.answer)
        correct       = verify(ground_truth, solver_answer)

        row_dict = dict(
            problem_id=row.problem_id,
            prompt=row.prompt,
            answer=ground_truth,
            solver_answer=solver_answer,
            raw_output=raw,
            has_boxed=has_boxed,
            correct=correct,
        )
        for col in _present_optional:
            row_dict[col] = getattr(row, col, None)
        results.append(row_dict)

    results_df = pd.DataFrame(results)

    # ── Report ───────────────────────────────────────────────────────────────
    n           = len(results_df)
    n_correct   = results_df['correct'].sum()
    adapter_acc = n_correct / n
    n_has_boxed = results_df['has_boxed'].sum()
    n_no_boxed  = n - n_has_boxed

    print(f'  ACCURACY (n={n}):  {adapter_acc:.4f}  ({n_correct}/{n})')
    print()
    print('Answer format breakdown:')
    print(f'  has \\boxed{{}}  : {n_has_boxed}/{n} ({n_has_boxed/n:.1%})')
    print(f'  no  \\boxed{{}}  : {n_no_boxed}/{n} ({n_no_boxed/n:.1%})  ← scored from fallback patterns')
    print()
    print('Accuracy by \\boxed{} presence:')
    print(
        results_df.groupby('has_boxed')['correct']
        .agg(['sum', 'count', 'mean'])
        .rename(columns={'sum': 'correct', 'count': 'total', 'mean': 'accuracy'})
        .to_string()
    )

    # ── Save ─────────────────────────────────────────────────────────────────
    save_cols = _core_cols + _present_optional
    save_df   = results_df[save_cols].copy()
    out_path  = out_path_for(adapter_name)
    save_df.to_parquet(out_path, index=False)
    print(f'\nSaved {len(save_df):,} rows → {out_path}')
    print('Sample outputs (first 3):')
    for _, r in results_df.head(3).iterrows():
        status = '✓' if r['correct'] else '✗'
        print(f'  [{status}] id={r["problem_id"]}  answer={r["answer"]!r}  solver_answer={r["solver_answer"]!r}')

    summary_rows.append(dict(adapter=adapter_name, accuracy=adapter_acc, n=n, n_correct=int(n_correct)))

# ── Cross-adapter summary ────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('  SUMMARY ACROSS ADAPTERS')
print(f'{"="*60}')
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))